# `ErrorAnalysis` module

The `bhm.ErrorAnalysis` module provides functions for assessing the error of an optimized model.

Note that these functions all asume the input model has been optimized, if given a non-optimized model,
the results will likely be confusing and in some way incorrect.

First lets load some data to use for demonstration.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import tables as tb
import smfbursts as smf

import H2MMbursts as bhm


raw = smf.photonHDF5.load('data/HP3_TE300_SPC630.hdf5')
data = smf.photonHDF5.regularize_dets(raw)

with tb.open_file('data/statepathparams.hdf5') as f:
    statepaths_hp3 = [smf.Param.decode_group(g) for g in
                      f.list_nodes(f.root.HP3_TE300_SPC630.statepath_params)]

## Loglik Error

The first way to assess the error, and the generally prefered way,
is to compute the point at which the loglikelihood drops by some amount when varying a particular parameter.


### Basic Evaluation

The most common way to evaluate the loglik error is to use the `bhm.error.statepath_ll_error`.
The basic signature is `bhm.error.statepath_ll_error(data, statepath, adjust='trans')`.

Simply provide the source `smf.PhotonData` or `smf.PhotonDataList` data object (`data` argument), 
and the `bhm.StatePath` (or one of it's subclasses) based `smf.Param` (`statepath` arguments,
and specify which array to adjust (`adjust` argument).

The return values are always the low and high limits of each array.
The return values are nested tuples, shaped according to the shape of the adjusted model array.
The values are the $\mathrm{H^{2}MM}$ models found with the values where the parameter in the given position
has a loglik 0.5 less than the input model.

In [2]:
pel, peh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], adjust='prior')
tel, teh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], adjust='trans')
oel, oeh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], adjust='obs')

In [3]:
model = statepaths_hp3[3].model
# get trans arrays of original and adjustes, showing lower bound
# use subtraction to show which values are adjusted
model.trans, tel[0][1].trans - model.trans

(array([[9.99968004e-01, 1.01524888e-05, 2.74626973e-06, 1.90968387e-05],
        [8.25172259e-06, 9.99969622e-01, 2.98876315e-06, 1.91374544e-05],
        [1.17852187e-06, 1.53085181e-06, 9.99980791e-01, 1.65000528e-05],
        [4.86084092e-06, 4.30962666e-06, 8.07403897e-06, 9.99982755e-01]]),
 array([[ 6.99860705e-07, -6.99860705e-07,  0.00000000e+00,
          0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00]]))

### Adjust functions

The core function for evaluating the loglikelihood error is `bhm.error.evalutate_ll_error`.
`bhm.error.statepath_ll_error` calls this function internally.

The basic method is to evaluate the loglikelihood for models, generated by an `adjust` function,
where a given parameter has been varied, until the loglikelihood of the adjusted model is
as set amount less than the optimal.
Adjust functions take a model and a floating point value that specify by how much to change the model.
Basic default `adjust`  functions are specified as string "prior", "trans" and "obs".

First the loglike of the input model is evaluated, and from this the target loglikelihood is determined.
This is used to generate a penatly function, where the returned value is the square distance $(ll_{adjusted} - ll_{target})^{2}$
of the adjusted model from the targed.
[scipy.optimize.fminbound](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fminbound.html)
is used to find this model.

Below is a simple example of how to evaluate the error of a model:

In [4]:
# from the statepath, get the necessary model/arrays
model = statepaths_hp3[3].params['model']
indexes = data.get_table(statepaths_hp3[3])['indexpath']
times = data.get_table(statepaths_hp3[3])['timepath']

# evaluate the error
terrl, terrh = bhm.error.evalutate_ll_error(model, indexes, times,
                                            bhm.error.trans_adjust,
                                            loc=(0,1), targ=0.5)
# display the low/high values of loc
print(terrl.trans[0,1], terrh.trans[0,1])

9.452623884694585e-06 1.087440595353463e-05


The basic signature is `bhm.error.evaluate_ll_error(model, indexes, times, adjust, targ=0.5)`,

**Core arguments**

1. `model` is the `hm.h2mm_model` to find the error of
2. `indexes` is the $\mathrm{H^{2}MM}$ index array ("indexpath" column of the `StatePath` `Param`)
3. `times` is the $\mathrm{H^{2}MM}$ index array ("timepath" column of the `StatePath` `Param`)
4. `adjust` a function with the signature `adjust(model, val, **kwargs)`

**optimization arguments**

It should also be noted that `bhm.error.evaluate_ll_error()` also has the `bound_kwargs`, passed to
[scipy.optimize.fminbound](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fminbound.html) 
keyword argument, as `scipy.optimize.fminbound` is used to find the model loglik decreased by `targ`.
The function given to `scipy.optimize.fminbound` evaluates $(ll_{opt-targ} - ll_{adj})^{2}$.

This requires calling `hm.h2mm_model.evaluate`, kwargs can be passed to this function through the `eval_kwargs` argument.

### Default `adjust` function kwargs

The `adjust` functions are called in the following way: `adjust(model, val, **kwargs)`

Note that any additional kwargs passed to `bhm.error.evaluate_ll_error()` are forwarded to the `adjust` function.

There are 3 built-in options for `adjust`:
1. `bhm.error.prior_adjust` (can also be speciffied as a string, `"prior"`)
2. `bhm.error.trans_adjust` (can also be speciffied as a string, `"trans"`)
3. `bhm.error.obs_adjust` (can also be speciffied as a string, `"obs"`)

In all three of these cases, they require the `loc` keyword argument.
This specifies the "location" (typically a tuple indexing into the specified array), of the parameter to adjust. 
The location may also be specified as a boolean mask, or sequence of locations if multiple location should be adjusted synchronysly. 
The locations should however all belong to the same row

This location specifies the parameter to be adjusted in the given direction.
However, because rows must be row-stochastic, all other parameters in the given row are adjusted in the opposite direction.
Also, adjustment by `val` is computed such that `val` exists in the interval $(0,1)$, rescaling all values accordingly,
and the lower bound is evaluate on the interval $(0, 0.5)$, and likewise the upper bound evaluated on the interval $(0.5, 1)$

#### `outer` argument

If, instead of adjusting all values in the row by a certain amount, it is also possible to specify the
`outer` keyword argument.
Here, pass a boolean mask , or sequence of locs of all "free" parameters. This must be a superset of locs.

In [5]:
# example as sequence of locs
bhm.error.obs_adjust(model, 0.3, loc=(0,1), outer=((0,0), (0,1))).obs - model.obs

array([[ 6.97925109e-02, -6.97925109e-02,  0.00000000e+00],
       [ 2.22044605e-16,  1.38777878e-17,  1.38777878e-17],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])

In [6]:
# example as boolean mask
outer = np.zeros((4,3), dtype=np.bool_)
outer[0,:2] = True
bhm.error.obs_adjust(model, 0.3, loc=(0,1), outer=outer).obs - model.obs

array([[ 6.97925109e-02, -6.97925109e-02,  0.00000000e+00],
       [ 2.22044605e-16,  1.38777878e-17,  1.38777878e-17],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])

#### Example: adjusting only values relevant to PR

Combining these we can now show how to evaluate just the error in $^{i}E_{raw}$, leaving $^{2}S_{raw}$.

Note that `bhm.error.statepath_ll_error()` has the `loc` keyword argument built-in, 
if it is not supplied (like the first example), then it iterates over every `loc` in the prior, trans or obs array.

Kwargs handed as kwargs to `bhm.error.evaluate_ll_error()` must be passed through the `adj_kwargs` keyword argument.

In [7]:
elow, ehigh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], 
                                           adjust='obs', loc=(0,1), 
                                           adj_kwargs={'outer':outer})

## BootStrap Error

An alternative to the loglik evaluation, which is generally simpler is the bootstrap method.
Here the data is divided into chuncs, and an independant optimization is conducted on each chunk.
The result is a new model for each chunck, 
and error is defned by the standard error of a given parameter based on the variance of that parameter in each chunck.

This is done useing the `bhm.error.BootStrapError.evaluate()` class method.

Simply pass the data and a `bhm.StatePath` based `smf.Param`, and it will produce a `bhm.BootStrapError` object, which stores the relavant errors.
The number of chuncks is specified with the keyword argument `n`, the default is `n=10`.

In [8]:
bserr = bhm.error.BootStrapError.evaluate(data, statepaths_hp3[3], n=10)

The model converged after 175 iterations
The model converged after 196 iterations
The model converged after 310 iterations
The model converged after 244 iterations
The model converged after 318 iterations
The model converged after 174 iterations
The model converged after 181 iterations
The model converged after 248 iterations
The model converged after 458 iterations
The model converged after 338 iterations


Accessing the error is done using the `err_prior/trans/obs` properties.

In [9]:
bserr.err_prior, bserr.err_trans, bserr.err_obs

(array([0.00605974, 0.00951431, 0.01199174, 0.0141201 ]),
 array([[1.16488527e-06, 9.19097866e-07, 4.87603212e-07, 1.48729007e-06],
        [7.92213736e-07, 2.22718315e-06, 6.49604626e-07, 2.61211196e-06],
        [1.55352459e-07, 3.12707825e-07, 1.00957434e-06, 1.00588659e-06],
        [2.77011885e-07, 4.55168225e-07, 5.73257125e-07, 8.19387164e-07]]),
 array([[0.07483015, 0.00221965, 0.07609387],
        [0.07440479, 0.001425  , 0.07434516],
        [0.00128171, 0.00170453, 0.0025162 ],
        [0.00200295, 0.00064494, 0.00183823]]))

The standard deviations can also be accessed through analagous
`std_prior/trans/obs` properties:

In [10]:
bserr.std_prior, bserr.std_trans, bserr.std_obs

(array([0.01916259, 0.03008689, 0.03792122, 0.04465167]),
 array([[3.68369067e-06, 2.90644265e-06, 1.54193674e-06, 4.70322417e-06],
        [2.50519980e-06, 7.04297153e-06, 2.05423020e-06, 8.26022331e-06],
        [4.91267612e-07, 9.88868968e-07, 3.19255440e-06, 3.18089268e-06],
        [8.75988496e-07, 1.43936831e-06, 1.81279820e-06, 2.59112972e-06]]),
 array([[0.2366337 , 0.00701915, 0.24062995],
        [0.23528859, 0.00450623, 0.23510003],
        [0.00405314, 0.00539021, 0.00795694],
        [0.00633388, 0.00203949, 0.00581299]]))

There are also methods `bhh.error.BootStrapError.col_std()` and `bhh.error.BootStrapError.col_error()`,
where a `smf.Column` object is handed as the only argument, and the function will internally
use `bhm.StatePathBase.model_value` to evaluate the expected error of the given `smf.Column`.
Note that this requires the `smf.Column` has a method for evaluating the expected value from a
$\mathrm{H^{2}MM}$ model.

In [11]:
E = smf.Column(statepaths_hp3[3].base_param, 'E_raw')

bserr.col_std(E), bserr.col_error(E)

(array([0.00000000e+00, 0.00000000e+00, 1.11022302e-16, 2.77555756e-17]),
 array([0.00000000e+00, 0.00000000e+00, 3.51083347e-17, 8.77708367e-18]))

Finally, it should be noted that the `bhm.error.BootStrapError` object stores the creating `smf.Param`
in the `bhm.error.BootStrapError.param` attribute, 
and each evaluated model, as a tuple in the `bhm.error.BootStrapError.models` attribute:

In [12]:
bserr.param.params

(('model',
  nstate: 4, ndet: 3, nphot: 0, niter: 0, loglik: -inf converged state: 0x8000
  prior:
  0.09180008464206446, 0.14192610716122736, 0.24635708795371888, 0.5199167202429893
  trans:
  0.9999680044027129, 1.0152488816594678e-05, 2.7462697332324526e-06, 1.909683873726998e-05
  8.25172258984347e-06, 0.9999696220598387, 2.9887631522342066e-06, 1.913745441912435e-05
  1.1785218691734586e-06, 1.530851813100379e-06, 0.9999807905734902, 1.6500052827459896e-05
  4.8608409182432066e-06, 4.309626662595637e-06, 8.074038970015473e-06, 0.9999827554934492
  obs:
  0.07804743236692008, 0.07009259344021249, 0.8518599741928675
  0.8592637589574251, 0.077001560515273, 0.06373468052730169
  0.153134712641781, 0.2986820101455996, 0.5481832772126195
  0.45271986441892176, 0.09304328947196071, 0.4542368461091176),
 ('streams',
  (<class 'smfbursts.ph_sel.PhSel'>
   <class 'smfbursts.ph_sel.PhStream'>
   ex = (True):(0)
   em = (True):(0)
   pol = all
   split = all,
   <class 'smfbursts.ph_sel.PhSe

In [13]:
bserr.models[-1]

nstate: 4, ndet: 3, nphot: 0, niter: 0, loglik: -inf converged state: 0x8000
prior:
0.0653970003803049, 0.18986032598527416, 0.29552763062575493, 0.44921504300866605
trans:
0.9999713910994743, 4.558384175118647e-06, 3.0470608800308154e-07, 2.3745810262527436e-05
6.024034190337315e-06, 0.9999663171541834, 6.086851885347634e-07, 2.7050126437628115e-05
6.275919587227391e-07, 1.0409572865311404e-06, 0.9999838124720684, 1.4518978686468071e-05
5.300324856855417e-06, 5.4113569083884675e-06, 6.316072272536246e-06, 0.9999829722459622
obs:
0.07933385726868852, 0.0774086951881182, 0.8432574475431933
0.8397670257923788, 0.07800587316723069, 0.08222710104039058
0.15708351322926892, 0.29401927789026916, 0.548897208880462
0.45018679140414847, 0.09393981393821182, 0.4558733946576398

## Discusion: Loglikelihood vs BootStrapError

Advantages of LL error
- No optimizations, can be faster to evaluate
- Decrease in loglikelihood provides clear threshold

Advantages of BootStrapError
- Evaluates all parameters together, so correlations can be more easily observed

In general the LL error is prefered because it is a more well defined evaluation.